In [ ]:
# Standard Library Imports
from abc import ABC, abstractmethod
import os
import time
from typing import Any, Dict, List, Tuple

# Data Processing & Visualization
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
sns.set_theme(style="whitegrid")

# Machine Learning & NLP
from datasets import load_dataset
import torch
import re
# import fasttext
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import joblib


In [ ]:
! pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 2.8 MB/s eta 0:00:00


In [ ]:
import re
import joblib
import pandas as pd
import numpy as np
from typing import Tuple, Dict, Any
from groq import Groq
from sklearn.pipeline import Pipeline

In [ ]:
class LanguageDetector:
    def __init__(self, model_path: str = "/content/language_detector.pkl"):
        try:
            self.pipeline = joblib.load(model_path)
            print(f"ML Model loaded successfully from {model_path}")
        except Exception:
            print("Warning: Model file not found. Please ensure it is trained and saved.")
            self.pipeline = None

    def predict_with_confidence(self, text: str) -> Tuple[str, float]:
        """Returns the predicted language code and its corresponding confidence score."""
        clean_text = str(text).strip()
        if not clean_text or self.pipeline is None:
            return "en", 1.0

        vectorizer = self.pipeline.named_steps['vectorizer']
        classifier = self.pipeline.named_steps['classifier']

        # Transform input text
        vec = vectorizer.transform([clean_text])

        # Safeguard for total Out-of-Vocabulary (OOV) text
        if vec.nnz == 0:
            return "unknown", 0.0

        # Extract class probabilities
        probs = classifier.predict_proba(vec)[0]
        classes = classifier.classes_

        max_idx = np.argmax(probs)
        pred_lang = str(classes[max_idx])
        confidence = float(probs[max_idx])

        return pred_lang, confidence

In [ ]:
# Initialize your Groq client (using your existing project stack)
GROQ_API_KEY = userdata.get('GROQ_API_KEY')
groq_client = Groq(api_key=GROQ_API_KEY)
ml_detector = LanguageDetector()

ML Model loaded successfully from /content/language_detector.pkl


In [ ]:
def validate_language(text: str, model_lang: str, confidence: float) -> str:
    """
    Validates language predictions using script-family groupings to prevent
    cross-language collisions (like Arabic vs. Urdu or Chinese vs. Japanese).
    """
    clean_text = text.strip().lower()
    if not clean_text:
        return 'en'


    # --- HINDI DEVANAGARI CHECK (Safe: Only 'hi' uses this in your dataset) ---
    if re.search(r'[\u0900-\u097F]', clean_text):
        return 'hi'

    # --- 4. LOW CONFIDENCE LATIN SCRIPT FALLBACK (e.g., Spanish vs. French) ---
    if (model_lang == 'en' and confidence < 0.70) or model_lang == 'unknown':
        print("I am in Low Confidence")
        print(f"  [Low Confidence]: Ambiguous Latin text ({confidence*100:.1f}%). Verifying...")

    print("I am going to groq")
    return query_llm_for_language_code(text)

def query_llm_for_language_code(text: str) -> str:
    """Helper function to offload ambiguous script decisions to Groq."""
    system_instruction = f"""
    You are a precise language identification API.
    Your task is to identify the ISO 639-1 two-letter code of the text provided.
    Respond ONLY with the 2-letter code (e.g., en, fr, es, ar, ur, zh, ja).
    """
    user_input = f"Text to analyze: \"{text}\""

    try:
        response = groq_client.chat.completions.create(
            model="llama3-8b-8192",
            messages=[
                {"role": "system", "content": system_instruction},
                {"role": "user", "content": user_input}
            ],
            temperature=0.0
        )
        validated_lang = response.choices[0].message.content.strip().lower()
        return validated_lang if len(validated_lang) <= 3 else "en"
    except Exception:
        return "en" # Secure fallback

In [ ]:
def translate_query_if_needed(text: str, language_code: str) -> str:
    """Translates non-English queries to English to protect RAG vector search alignment."""
    if language_code == 'en':
        return text

    print(f"--> Translating query from '{language_code}' to English for Vector Database Search alignment...")
    translation_prompt = f"""
    Translate the following user mental health query into clear, natural English.
    Preserve all original emotional weight and semantic nuance.
    Return ONLY the translated text. Do not add introductions or explanations.

    Query: "{text}"
    """
    try:
        response = groq_client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[{"role": "user", "content": translation_prompt}],
            temperature=0.0
        )
        return response.choices[0].message.content.strip()
    except Exception:
        return text # Fallback to original text if API fails

In [ ]:
# Simulation Helper Function
def run_pipeline(user_input: str):
    print(f"\n[Raw Input]: \"{user_input}\"")

    # 1. Run traditional ML model
    pred_lang, confidence = ml_detector.predict_with_confidence(user_input)
    print(f"  -> ML Model Output: '{pred_lang}' with {confidence*100:.1f}% confidence.")

    # 2. Run the Validation Layer
    final_lang = validate_language(user_input, pred_lang, confidence)
    print(f"  -> Final Validated Language: '{final_lang}'")

    # 3. Translate if required
    search_query = translate_query_if_needed(user_input, final_lang)
    print(f"[Vector DB Search Query]: \"{search_query}\"")
    print("-" * 60)


In [ ]:
# =====================================================================
# TEST CASES
# =====================================================================

# Test Case 1: Clear English (High Confidence Track)
run_pipeline("I feel incredibly stressed about my upcoming presentation.")

# Test Case 2: Short text or OOV non-Latin (Script Override Track)
run_pipeline("أشعر بالقلق الشديد")

# Test Case 3: Tricky Latin-based non-English text that might cause low confidence
run_pipeline("Je ne me sens pas bien du tout aujourd'hui.")


[Raw Input]: "I feel incredibly stressed about my upcoming presentation."
  -> ML Model Output: 'en' with 39.5% confidence.
I am in Low Confidence
  [Low Confidence]: Ambiguous Latin text (39.5%). Verifying...
I am going to groq
  -> Final Validated Language: 'en'
[Vector DB Search Query]: "I feel incredibly stressed about my upcoming presentation."
------------------------------------------------------------

[Raw Input]: "أشعر بالقلق الشديد"
  -> ML Model Output: 'ar' with 72.0% confidence.
I am in Arabic
  -> Final Validated Language: 'ar'
--> Translating query from 'ar' to English for Vector Database Search alignment...
[Vector DB Search Query]: ""I'm feeling extremely anxious.""
------------------------------------------------------------

[Raw Input]: "Je ne me sens pas bien du tout aujourd'hui."
  -> ML Model Output: 'fr' with 94.4% confidence.
I am going to groq
  -> Final Validated Language: 'en'
[Vector DB Search Query]: "Je ne me sens pas bien du tout aujourd'hui."
--------

In [ ]:
run_pipeline("How are u?")


[Raw Input]: "How are u?"
  -> ML Model Output: 'zh' with 19.7% confidence.
I am going to groq
  -> Final Validated Language: 'en'
[Vector DB Search Query]: "How are u?"
------------------------------------------------------------
